In [1]:
# Wspolne definicje (jak w 20260716a/b/c.ipynb i 20260819a.ipynb - kopiowane,
# nie importowane, zeby ten notebook byl samodzielny). Cel: KOLEJNY krok z
# planu Homoli (patrz 20260819.txt pkt 4/18) - rozszerzenie pelnego skanu po
# t0 (P_days=3350, krok 6h, bez laga) z Moskwy/Oulu na pozostale 20 stacji
# NMDB.
#
# Status watku replikacji headline'owego wyniku Moskwy (2026-08-20): cel
# tamtej pracy byla KALIBRACJA METODOLOGII - sprawdzenie, ze nasz pipeline
# (cosmoseismic_stat ponizej) liczy to samo co pipeline zespolu/artykulu.
# To sie udalo: po naprawie regresji pandas 3.0 (origin="start"), kod
# zespolu (pdf_values6/8.py z credo-internal) odpalony na ich danych
# odtwarza ICH WLASNY zapisany wynik (214/118) BIT-FOR-BIT (20260819.txt
# pkt 14), a sposob binowania/kompletnosc filtra rownania (3) okazaly sie
# bez znaczenia (pkt 18) - reszta rozbieznosci do publikacji (218/113) to
# wylacznie roznica konkretnej wersji pliku CR Moskwy, NIE metodologii.
# Kalibracja = pipeline ponizej jest zwalidowany, uzywany dalej bez zmian.
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binom, norm

sys.path.insert(0, "..")
from mc_parallel import run_t0_scan_parallel

USGS_PATH = "../data/usgs_data/usgs_m4_2005_2025.csv"

def load_earthquakes(min_mag=4.0):
    df = pd.read_csv(USGS_PATH, usecols=["time", "mag"])
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_localize(None)
    df = df[df["mag"] >= min_mag]
    return df.set_index("time")["mag"].sort_index()


def cosmoseismic_stat(cr, eq, t0, P_days, d_days, m, dt_days):
    N = int(P_days // d_days)
    edges = pd.date_range(t0, periods=N + 1, freq=pd.Timedelta(days=d_days))
    eq_edges = edges + pd.Timedelta(days=dt_days)

    cr_cats = pd.cut(cr.index, edges, right=False)
    cr_binned = cr.groupby(cr_cats, observed=False).mean().reindex(cr_cats.categories)
    cr_vals = cr_binned.to_numpy()

    eq_in_range = eq[(eq.index >= eq_edges[0]) & (eq.index < eq_edges[-1])]
    eq_cats = pd.cut(eq_in_range.index, eq_edges, right=False)
    eq_binned = eq_in_range.groupby(eq_cats, observed=False).sum().reindex(eq_cats.categories, fill_value=0.0)
    sm_vals = eq_binned.to_numpy()

    nCR_i, nCR_im1 = cr_vals[1:], cr_vals[:-1]
    dCR = nCR_i - nCR_im1
    Sm = sm_vals[1:]

    med_Sm = np.nanmedian(Sm)
    med_dCR = np.nanmedian(np.abs(dCR))

    A = Sm / med_Sm - 1
    B = np.abs(dCR) / med_dCR - 1

    valid = (
        (A != 0) & (B != 0) &
        (nCR_i > 0) & (nCR_im1 > 0) &
        (Sm > 0) &
        ~np.isnan(A) & ~np.isnan(B)
    )

    c_valid = (A * B)[valid]
    Np, Nm = int((c_valid > 0).sum()), int((c_valid < 0).sum())
    n_total = Np + Nm

    if n_total == 0:
        return dict(N=N, N_valid=0, Np=0, Nm=0, PPDF=np.nan, PCDF=np.nan, sigma=np.nan)

    ppdf = binom.pmf(Np, n_total, 0.5)
    pcdf = binom.sf(Np - 1, n_total, 0.5)
    sigma = norm.isf(pcdf)

    return dict(N=N, N_valid=n_total, Np=Np, Nm=Nm, PPDF=ppdf, PCDF=pcdf, sigma=sigma)


eq = load_earthquakes(min_mag=4.0)
print(f"EQ (M>=4.0): {len(eq)} zdarzen, {eq.index.min()} .. {eq.index.max()}")

EQ (M>=4.0): 290945 zdarzen, 2005-01-01 00:47:34.620000 .. 2025-01-31 23:57:39.481000


In [2]:
# Loadery CR - Moskwa/Oulu jak w 20260819a.ipynb (dedykowane pliki pelnej
# historii), pozostale 20 stacji z NOWEGO pobrania
# `data/csv_data_stations_full6h/{stacja}_full_6h.csv`
# (skrypt: skrypty_dl/wlasne/download_stations_full_range.py, NIE
# `csv_data_stations_native6h/` - ten ostatni to tylko wycinek 2011-2019,
# za krotki dla P_days=3350 na pelnym zakresie kandydatow t0 2005-2015,
# patrz uzasadnienie w docstringu skryptu pobierajacego).
import os

MOSC_PATH = "../data/mosc_data.csv"
OULU_PATH = "../data/oulu_5min_data.csv"
FULL6H_DIR = "../data/csv_data_stations_full6h"

OTHER_STATIONS = [
    "THUL", "LMKS", "APTY", "SOPB", "JUNG1", "SOPO", "FSMT",
    "JUNG", "NEWK", "PWNK", "MXCO", "NAIN", "HRMS", "TERA", "INVK", "ATHN",
    "AATB", "PSNM", "NANM", "MCRL",
]


def load_mosc():
    df = pd.read_csv(MOSC_PATH)
    df["datetime"] = pd.to_datetime(df["datetime"])
    return df.set_index("datetime").sort_index()["value"]


def load_oulu():
    # Loader jak w 20260716b/c.ipynb / 20260819a.ipynb: oulu_5min_data.csv ma
    # jeden uszkodzony wiersz (linia 839815) - pomijamy przez errors="coerce"
    # + dropna.
    df = pd.read_csv(OULU_PATH)
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    n_bad = df["datetime"].isna().sum()
    if n_bad:
        print(f"load_oulu: pominieto {n_bad} niesparsowalnych wierszy (uszkodzone dane)")
    df = df.dropna(subset=["datetime"])
    return df.set_index("datetime").sort_index()["value"]


def load_station_full6h(station):
    path = os.path.join(FULL6H_DIR, f"{station.lower()}_full_6h.csv")
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    return df.set_index("datetime").sort_index()["value"]


cr_series = {}
cr_series["mosc"] = load_mosc()
cr_series["oulu"] = load_oulu().resample("6h").mean()
print(f"mosc: {len(cr_series['mosc'])} pomiarow, {cr_series['mosc'].index.min()} .. {cr_series['mosc'].index.max()}")
print(f"oulu: {len(cr_series['oulu'])} pomiarow (po resample 6h), {cr_series['oulu'].index.min()} .. {cr_series['oulu'].index.max()}")

missing_stations = []
for station in OTHER_STATIONS:
    path = os.path.join(FULL6H_DIR, f"{station.lower()}_full_6h.csv")
    if not os.path.exists(path):
        missing_stations.append(station)
        continue
    s = load_station_full6h(station)
    cr_series[station.lower()] = s
    print(f"{station.lower()}: {len(s)} pomiarow, {s.index.min()} .. {s.index.max()}")

if missing_stations:
    print(f"\nBRAK plikow dla {len(missing_stations)} stacji (jeszcze nie pobrane): {missing_stations}")
    print("Uruchom najpierw: python3 skrypty_dl/wlasne/download_stations_full_range.py")


load_oulu: pominieto 1 niesparsowalnych wierszy (uszkodzone dane)
mosc: 91824 pomiarow, 1960-01-01 00:00:00 .. 2025-03-23 18:00:00
oulu: 81816 pomiarow (po resample 6h), 1970-01-01 00:00:00 .. 2025-12-31 18:00:00
thul: 30628 pomiarow, 2005-01-01 00:00:00 .. 2025-12-31 18:00:00
lmks: 27056 pomiarow, 2005-01-01 00:00:00 .. 2023-07-10 18:00:00
apty: 30665 pomiarow, 2005-01-01 00:00:00 .. 2025-12-31 18:00:00
sopb: 23152 pomiarow, 2005-01-01 00:00:00 .. 2025-12-31 18:00:00
jung1: 30529 pomiarow, 2005-01-01 00:00:00 .. 2025-12-31 18:00:00
sopo: 24434 pomiarow, 2005-01-01 00:00:00 .. 2025-12-31 18:00:00
fsmt: 30522 pomiarow, 2005-01-01 00:00:00 .. 2025-12-31 18:00:00
jung: 30630 pomiarow, 2005-01-01 00:00:00 .. 2025-12-31 18:00:00
newk: 30491 pomiarow, 2005-01-01 00:00:00 .. 2025-12-31 18:00:00
pwnk: 28074 pomiarow, 2005-01-01 00:00:00 .. 2025-12-31 18:00:00
mxco: 29602 pomiarow, 2005-01-01 00:00:00 .. 2025-10-14 12:00:00
nain: 30394 pomiarow, 2005-01-01 00:00:00 .. 2025-12-31 18:00:00
hrms: 

In [3]:
# Parametry skanu - TE SAME co w 20260819a.ipynb (pelna konfiguracja
# uzgodniona z Homola, komentarz 2026-07-24 / odpowiedz Maćka 2026-08-15):
#   P_days=3350, krok t0=6h, d w {1,5}, dt_days=0 (bez laga), m=4.0.
# Zakres kandydatow t0 identyczny dla WSZYSTKICH stacji (wyznaczony tylko
# przez katalog EQ) - stacje z krotszym pokryciem CR beda mialy mniej
# WAZNYCH kandydatow (odrzucanych przez filtr NaN w cosmoseismic_stat), ale
# same t0_candidates sa wspolne, zeby wyniki byly porownywalne 1:1.
P_DAYS = 3350
D_VALUES = [1, 5]
DT_DAYS = 0
M_THRESHOLD = 4.0

t0_first = eq.index.min().ceil("6h")
t0_last = (eq.index.max() - pd.Timedelta(days=P_DAYS)).floor("6h")
t0_candidates = pd.date_range(t0_first, t0_last, freq="6h")

print(f"Zakres kandydatow t0: {t0_first} .. {t0_last}")
print(f"Liczba kandydatow t0: {len(t0_candidates)}")


Zakres kandydatow t0: 2005-01-01 06:00:00 .. 2015-11-30 18:00:00
Liczba kandydatow t0: 15943


In [4]:
# Przycinanie do zakresu EQ (wydajnosc - patrz komentarz w 20260819a.ipynb:
# cosmoseismic_stat robi pd.cut na calym cr.index przy kazdym wywolaniu).
def clip_to_eq_range(series):
    return series[(series.index >= eq.index.min()) & (series.index <= eq.index.max())]

cr_clipped = {name: clip_to_eq_range(s) for name, s in cr_series.items()}

for name, s in cr_clipped.items():
    frac_covered = len(s) / len(t0_candidates) if len(t0_candidates) else float("nan")
    print(f"{name}: {len(cr_series[name])} -> {len(s)} po przycieciu "
          f"({s.index.min() if len(s) else 'brak'} .. {s.index.max() if len(s) else 'brak'})")


mosc: 91824 -> 28979 po przycieciu (2005-01-01 06:00:00 .. 2025-01-31 18:00:00)
oulu: 81816 -> 29343 po przycieciu (2005-01-01 06:00:00 .. 2025-01-31 18:00:00)
thul: 30628 -> 29309 po przycieciu (2005-01-01 06:00:00 .. 2025-01-31 18:00:00)
lmks: 27056 -> 27055 po przycieciu (2005-01-01 06:00:00 .. 2023-07-10 18:00:00)
apty: 30665 -> 29337 po przycieciu (2005-01-01 06:00:00 .. 2025-01-31 18:00:00)
sopb: 23152 -> 21848 po przycieciu (2005-01-01 06:00:00 .. 2025-01-31 00:00:00)
jung1: 30529 -> 29192 po przycieciu (2005-01-01 06:00:00 .. 2025-01-31 18:00:00)
sopo: 24434 -> 23130 po przycieciu (2005-01-01 06:00:00 .. 2025-01-31 00:00:00)
fsmt: 30522 -> 29195 po przycieciu (2005-01-01 06:00:00 .. 2025-01-31 18:00:00)
jung: 30630 -> 29293 po przycieciu (2005-01-01 06:00:00 .. 2025-01-31 18:00:00)
newk: 30491 -> 29162 po przycieciu (2005-01-01 06:00:00 .. 2025-01-31 18:00:00)
pwnk: 28074 -> 26741 po przycieciu (2005-01-01 06:00:00 .. 2025-01-31 18:00:00)
mxco: 29602 -> 28605 po przycieciu (200

In [ ]:
# Pelny skan po t0 dla WSZYSTKICH dostepnych stacji (mosc/oulu + te z
# OTHER_STATIONS ktore maja juz sciagniete dane), osobno dla d=1 i d=5.
# Ten sam wzorzec co 20260819a.ipynb komorki 4-5 (run_t0_scan_parallel z
# mc_parallel.py), tylko w petli po stacjach. Zapis od razu do
# results/t0_scan_{stacja}_d{d}.csv (niewersjonowane w git).
#
# UWAGA: przy 22 stacjach x 2 wartosci d to jest duzo obliczen - dla
# Moskwy/Oulu (20260819a.ipynb) d=1 zajelo ~309s/stacje, d=5 ~68s/stacje
# (patrz 20260819.txt pkt 5). Dla pozostalych 20 stacji nalezy sie
# spodziewac podobnego rzedu wielkosci (moze mniej, bo krotsze pokrycie CR
# = mniej wazny okien do policzenia per kandydat) - w sumie rzedu 1-2h
# obliczen. NIE odpalac tej komorki bez zastanowienia nad czasem.
scan_results = {}
for name, cr in cr_clipped.items():
    if len(cr) == 0:
        print(f"{name}: BRAK danych CR w zakresie EQ, pomijam skan")
        continue
    scan_results[name] = {}
    for d in D_VALUES:
        print(f"\n=== {name}, d={d} ===")
        t_start = time.time()
        df_scan = run_t0_scan_parallel(
            cr, eq, t0_candidates,
            P_days=P_DAYS, d_days=d, m=M_THRESHOLD, dt_days=DT_DAYS,
            stat_fn=cosmoseismic_stat,
            save_path=f"../results/t0_scan_{name}_d{d}.csv",
        )
        n_valid = df_scan["N_valid"].gt(0).sum() if "N_valid" in df_scan else None
        print(f"gotowe w {time.time() - t_start:.1f}s, {len(df_scan)} wierszy, "
              f"{n_valid} z nieproznym N_valid")
        scan_results[name][d] = df_scan


In [ ]:
# Ekstremalny PPDF (definicja "sily efektu" wg Homoli - patrz 20260819a.ipynb
# komorka 6) dla kazdej stacji/d, porownany z benchmarkiem log10(PPDF)<-8.
# Wazna zastrzezenie z 20260819.txt pkt 6c (efekt "look elsewhere"
# nieskorygowany) dotyczy TEZ tego skanu - ~16000 silnie skorelowanych
# kandydatow t0 na stacje, wiec pojedyncze przekroczenie benchmarku na
# ktorejs z 22 stacji NIE jest samo w sobie dowodem efektu, tylko sygnalem
# do dalszego sprawdzenia (analogicznie do wyniku Oulu d=5 z 19.08).
HOMOLA_LOG10_PPDF_BENCHMARK = -8

rows = []
for name, results in scan_results.items():
    for d, df_scan in results.items():
        valid_scan = df_scan.dropna(subset=["PPDF"])
        if len(valid_scan) == 0:
            rows.append(dict(station=name, d=d, t0_best=pd.NaT, Np=np.nan, Nm=np.nan,
                              PPDF=np.nan, PCDF=np.nan, sigma=np.nan, log10_PPDF=np.nan,
                              n_valid_candidates=0, beats_homola_benchmark=False))
            continue
        best = valid_scan.loc[valid_scan["PPDF"].idxmin()]
        rows.append(dict(
            station=name, d=d, t0_best=best["t0"],
            Np=int(best["Np"]), Nm=int(best["Nm"]),
            PPDF=best["PPDF"], PCDF=best["PCDF"], sigma=best["sigma"],
            log10_PPDF=np.log10(best["PPDF"]) if best["PPDF"] > 0 else -np.inf,
            n_valid_candidates=len(valid_scan),
            beats_homola_benchmark=(best["PPDF"] > 0 and np.log10(best["PPDF"]) < HOMOLA_LOG10_PPDF_BENCHMARK),
        ))

summary = pd.DataFrame(rows).sort_values(["d", "log10_PPDF"])
pd.set_option("display.width", 160)
print(summary.to_string(index=False))
print(f"\nBenchmark Homoli: log10(PPDF) < {HOMOLA_LOG10_PPDF_BENCHMARK}")
print(f"Stacje bijace benchmark: {summary.loc[summary['beats_homola_benchmark'], 'station'].tolist()}")

summary.to_csv("../results/t0_scan_22stations_summary.csv", index=False)


station  d             t0_best   Np   Nm         PPDF         PCDF     sigma  log10_PPDF  n_valid_candidates  beats_homola_benchmark
   athn  1 2008-10-25 18:00:00 1808 1477 7.785316e-10 4.146893e-09  5.762376   -9.108724               15943                    True
   psnm  1 2008-09-25 18:00:00 1827 1512 4.780504e-09 2.695550e-08  5.437904   -8.320526               15943                    True
   nain  1 2008-10-31 12:00:00 1815 1502 5.252773e-09 2.960717e-08  5.421158   -8.279611               15943                    True
   invk  1 2008-11-18 18:00:00 1782 1494 4.374555e-08 2.618992e-07  5.017384   -7.359066               15943                   False
   mxco  1 2008-11-25 00:00:00 1795 1508 5.273822e-08 3.189889e-07  4.979351   -7.277875               15943                   False
   fsmt  1 2008-10-22 18:00:00 1815 1529 6.657913e-08 4.083508e-07  4.931334   -7.176662               15943                   False
   hrms  1 2008-11-27 00:00:00 1811 1527 7.735179e-08 4.765192e-07  4

In [ ]:
# Wykres podsumowujacy: -log10(PPDF) ekstremum per stacja, osobno d=1/d=5,
# posortowane wg d=5 (glowna szerokosc binu artykulu), z linia progu
# referencyjnego. Zamiast 22 osobnych wykresow PPDF-vs-t0 (jak dla Moskwy/Oulu w
# 20260819a.ipynb) - tu chodzi o porownanie MIEDZY stacjami, analogicznie do
# tabeli "22 stacje" z 20260719b.ipynb. Etykiety celowo neutralne (bez
# nazwisk) - wykres przeznaczony do udostepniania zespolowi.
pivot = summary.pivot(index="station", columns="d", values="log10_PPDF")
order = pivot[5].sort_values().index if 5 in pivot.columns else pivot.index
pivot = pivot.loc[order]

fig, ax = plt.subplots(figsize=(10, 8))
y = np.arange(len(pivot))
width = 0.35
for i, d in enumerate(D_VALUES):
    if d not in pivot.columns:
        continue
    ax.barh(y + (i - 0.5) * width, -pivot[d], height=width, label=f"d={d}")
ax.axvline(-HOMOLA_LOG10_PPDF_BENCHMARK, color="gray", ls=":", label="próg referencyjny (PPDF=1e-8)")
ax.set_yticks(y)
ax.set_yticklabels(pivot.index)
ax.set_xlabel("-log10(PPDF) ekstremum skanu")
ax.set_title(f"Pelny skan po t0 (P_days={P_DAYS}, krok=6h, lag=0): sila efektu per stacja")
ax.legend()
ax.grid(ls=":", alpha=0.5, axis="x")
fig.tight_layout()
fig.savefig("../results/t0_scan_22stations_summary_plot.png", dpi=120)
plt.show()
